In [16]:
import os
import signal
from idlelib import history
from typing import Any

import torch
import dnnlpy
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as utils
import torchvision.datasets as datasets
import torchvision.transforms.v2 as v2
from torch import Tensor
from torchmetrics import Metric
from torch.distributed.elastic import metrics
from torch.types import Device
from torchmetrics.classification import MulticlassAccuracy

print("PyTorch version: ", torch.__version__)


PyTorch version:  2.13.0+cpu


In [ ]:
"""
为什么只保存模型参数还不够
    model.state_dict() 没有优化器中信息
    optimizer.state_dict()
"""

In [17]:
"""
训练一个简单的MLP
"""

dnnlpy.set_seed(42)
device = dnnlpy.get_default_device()

print("Using device: ", device)

root = dnnlpy.get_data_root()
transforms = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
ds_rng = torch.Generator().manual_seed(42)

train_ds = datasets.MNIST(root=root, train=True, transform=transforms, download=True)

trains_ds, val_ds = utils.random_split(train_ds, [50000, 10000], generator=ds_rng)
test_ds = datasets.MNIST(root=root, train=False, transform=transforms, download=True)

train_dl = utils.DataLoader(trains_ds, batch_size=64, shuffle=True)
val_dl = utils.DataLoader(val_ds, batch_size=128, shuffle=False)
test_dl = utils.DataLoader(test_ds, batch_size=128, shuffle=False)


# 定义一个MLP

class MLP(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)


#  创建训练对象
def create_training_objects(lr: float = 1e-3) -> tuple[
    nn.Module,
    nn.Module,
    optim.Optimizer,
    MulticlassAccuracy,
    MulticlassAccuracy
]:
    model = MLP(num_classes=10).to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    metric = MulticlassAccuracy(num_classes=10).to(device)
    val_metric = MulticlassAccuracy(num_classes=10).to(device)
    return model, loss_fn, optimizer, metric, val_metric


# 保存checkpoint

def save_checkpoint(path: str | os.PathLike[str], epoch: int, model: nn.Module, optimizer: optim.Optimizer,
                    history: list[dict[str, float]]) -> None:
    checkpoint = {
        'epoch': epoch,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'history': history

    }
    torch.save(checkpoint, path)


# 加载checkpoint

def load_checkpoint(path: str | os.PathLike[str], model: nn.Module, optimizer: optim.Optimizer,
                    device: Device = None) -> tuple[int, list[dict[str, float]]]:
    if device is None:
        device = dnnlpy.get_default_device()
    else:
        device = torch.device(device)

    checkpoint = torch.load(path, map_location=device, weights_only=True)

    model.load_state_dict(checkpoint['model'])
    optimizer.load_state_dict(checkpoint['optimizer'])

    epoch = checkpoint['epoch']
    history = checkpoint['history']
    return epoch, history



Using device:  cpu


In [18]:
"""
第一次训练:模拟程序突然崩溃
"""
checkpoint_path = 'mnist-mlp-checkpoint.pt'
device = dnnlpy.get_default_device()

model = MLP(num_classes=10).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
metric = MulticlassAccuracy(num_classes=10).to(device)
val_metric = MulticlassAccuracy(num_classes=10).to(device)

history = []


def train_one_epoch(
        model: nn.Module,
        dataloader: torch.utils.data.DataLoader[tuple[Tensor, Tensor]],
        loss_fn: nn.Module,
        optimizer: optim.Optimizer,
        metric: Metric,
        device: torch.device,
) -> tuple[float, float]:
    model.train()
    metric.reset()
    total_loss = 0.0

    for X, y in dataloader:
        X = X.to(device)
        y = y.to(device)

        logits = model(X)
        loss = loss_fn(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        metric.update(logits.detach(), y)
    avg_loss = total_loss / len(dataloader)
    avg_metric = metric.compute().item()
    return avg_loss, avg_metric


def evaluate(model: nn.Module, dataloader: torch.utils.data.DataLoader[tuple[Tensor, Tensor]], loss_fn: nn.Module,
             metric: Metric, device: torch.device) -> tuple[float, float]:
    model.eval()
    metric.reset()

    total_loss = 0.0

    with  torch.inference_mode():
        for X, y in dataloader:
            X = X.to(device)
            y = y.to(device)

            logits = model(X)
            loss = loss_fn(logits, y)

            total_loss += loss.item()
            metric.update(logits, y)
    avg_loss = total_loss / len(dataloader)
    avg_metric = metric.compute().item()
    return avg_loss, avg_metric


num_epochs = 5
crash_after_epochs = 3

for epoch in range(1, num_epochs + 1):
    loss, acc = train_one_epoch(model=model, dataloader=train_dl, loss_fn=loss_fn, optimizer=optimizer, metric=metric,
                                device=device)
    val_loss, val_acc = evaluate(model=model, dataloader=val_dl, loss_fn=loss_fn, metric=metric, device=device)

    record = {
        'epoch': epoch,
        'loss': loss,
        'acc': acc,
        'val_loss': val_loss,
        'val_acc': val_acc
    }
    history.append(record)

    n = len(str(num_epochs))
    print(f"Epoch [{epoch:0{n}d}/{num_epochs:0{n}d}]"
          f'| loss:{loss:.4f}'
          f'| acc:{acc:.4f}'
          f'|val_loss:{val_loss:.4f}'
          f'|val_acc:{val_acc:.4f}')

    save_checkpoint(path=checkpoint_path, epoch=epoch, model=model, optimizer=optimizer, history=history)

    if epoch == crash_after_epochs:
        try:
            signal.raise_signal(signal.SIGINT)
        except  KeyboardInterrupt:
            print("ERROR - keyboard interrupt")
            break



Epoch [1/5]| loss:0.3255| acc:0.9105|val_loss:0.1886|val_acc:0.9465
Epoch [2/5]| loss:0.1388| acc:0.9598|val_loss:0.1399|val_acc:0.9588
Epoch [3/5]| loss:0.0912| acc:0.9728|val_loss:0.1045|val_acc:0.9682
ERROR - keyboard interrupt


In [19]:
"""
重新创建模型和优化器
"""
model = MLP(num_classes=10).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
metric = MulticlassAccuracy(num_classes=10).to(device)
val_metric = MulticlassAccuracy(num_classes=10).to(device)

last_epoch,history =  load_checkpoint(checkpoint_path,model,optimizer,device)

print("last finished epoch:",last_epoch)
print("Number of history records:",len(history))


"""
从checkpoint继续训练
"""

for epoch in range(last_epoch+1, num_epochs + 1):
    loss, acc = train_one_epoch(model=model, dataloader=train_dl, loss_fn=loss_fn, optimizer=optimizer, metric=metric,
                                device=device)
    val_loss, val_acc = evaluate(model=model, dataloader=val_dl, loss_fn=loss_fn, metric=metric, device=device)

    record = {
        'epoch': epoch,
        'loss': loss,
        'acc': acc,
        'val_loss': val_loss,
        'val_acc': val_acc
    }
    history.append(record)

    n = len(str(num_epochs))
    print(f"Epoch [{epoch:0{n}d}/{num_epochs:0{n}d}]"
          f'| loss:{loss:.4f}'
          f'| acc:{acc:.4f}'
          f'|val_loss:{val_loss:.4f}'
          f'|val_acc:{val_acc:.4f}')

    save_checkpoint(path=checkpoint_path, epoch=epoch, model=model, optimizer=optimizer, history=history)

"""
最终结果
"""
test_metric = MulticlassAccuracy(num_classes=10).to(device)
test_loss,test_acc= evaluate(model=model, dataloader=test_dl, loss_fn=loss_fn, metric=metric, device=device)
model_name =  model.__class__.__name__
print(f'[{model_name}|test_loss:{test_loss:.4f}|test_acc:{test_acc:.4f}]')

last finished epoch: 3
Number of history records: 3
Epoch [4/5]| loss:0.0662| acc:0.9808|val_loss:0.0923|val_acc:0.9724
Epoch [5/5]| loss:0.0502| acc:0.9851|val_loss:0.0944|val_acc:0.9715
[MLP|test_loss:0.0726|test_acc:0.9772]
